# NORIA — Stage 2 proof v2 (LivePortrait)

SadTalker is broken on modern Colab, so this uses **LivePortrait** (KwaiVGI) —
a well-maintained model that installs cleanly and produces **photoreal facial
animation** of Noria.

**Honest note:** LivePortrait is *motion-driven* — it animates Noria's face using
a short reference motion clip (included in the repo), not her own voice. This
proves photoreal Noria animation works, free. Audio lip-sync from her voice is a
further model.

**Run:** set GPU (`Runtime → Change runtime type → T4 GPU`), then `Runtime → Run
all`. ~5–8 min. If a cell errors, paste it to Claude.

## 0. Confirm the free GPU is on

In [ ]:
!nvidia-smi

## 1. Install LivePortrait + download its weights
_(a few minutes)_

In [ ]:
%cd /content
!git clone -q https://github.com/KwaiVGI/LivePortrait
%cd /content/LivePortrait
!pip install -r requirements.txt
from huggingface_hub import snapshot_download
snapshot_download('KwaiVGI/LivePortrait', local_dir='pretrained_weights', allow_patterns=['*.pth','*.onnx','*.safetensors','*.json','*.txt'])
print('LivePortrait ready')

## 2. Get Noria's face
Pulls her real render from your live site. For Noria-M use `noria-m.png`.

In [ ]:
!wget -q -O /content/noria.png https://noria-body.onrender.com/assets/noria-f.png
from IPython.display import Image
Image('/content/noria.png', width=280)

## 3. Animate Noria
_(uses a built-in reference motion clip; ~1–2 min)_

In [ ]:
import glob, os
os.chdir('/content/LivePortrait')
drivers = sorted(glob.glob('assets/examples/driving/*.mp4'))
print('available motion clips:', [os.path.basename(d) for d in drivers][:12])
driver = drivers[0]
print('using:', driver)
!python inference.py -s /content/noria.png -d {driver}

## 4. Watch her — and download

In [ ]:
import glob, os
from IPython.display import HTML
from base64 import b64encode
outs = sorted(glob.glob('/content/LivePortrait/animations/*.mp4'), key=os.path.getmtime)
assert outs, 'No video produced — check the previous cell output.'
final = [o for o in outs if 'concat' in o] or outs
mp4 = final[-1]
print('Animated Noria:', mp4)
data = b64encode(open(mp4,'rb').read()).decode()
display(HTML(f'<video width=400 controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))
try:
    from google.colab import files; files.download(mp4)
except Exception:
    print('Download from the Files panel:', mp4)